In [1]:
# Cài đặt thư viện mới nhất từ PyPI
# -q (quiet): Giảm bớt thông báo cài đặt dài dòng
# -U (upgrade): Đảm bảo cập nhật lên bản mới nhất
%pip install -q -U google-genai pandas python-dotenv

print("✅ Bước 1 hoàn tất: Các thư viện đã được cài đặt thành công!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 17.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
✅ Bước 1 hoàn tất: Các thư viện đã đư

In [2]:
import os
import getpass

# Biến cờ (flag) kiểm soát chế độ: True = Dùng phản hồi giả lập, False = Gọi API thật
USE_MOCK = False
api_key = None

# --- 1. Thử lấy key từ Colab Secrets (Cách chuẩn trên Google Colab) ---
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
    if api_key:
        print("🔑 Tuyệt vời! Đã nạp thành công GEMINI_API_KEY từ Colab Secrets.")
except Exception:
    # Đoạn này kích hoạt nếu không chạy trên Colab hoặc chưa cài Colab Secrets
    pass

# --- 2. Thử tìm từ biến môi trường hệ thống ---
if not api_key:
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# --- 3. Nếu vẫn chưa có, cho phép nhập bảo mật qua getpass ---
if not api_key:
    print("ℹ️ Chưa tìm thấy API key trong Secrets.")
    user_input = getpass.getpass("👉 Hãy dán GEMINI_API_KEY của bạn vào đây rồi bấm Enter (hoặc bấm Enter để trống nếu muốn dùng Mock Mode): ")
    if user_input.strip():
        api_key = user_input.strip()
        print("🔑 Đã lưu nhận API key từ bạn!")
    else:
        USE_MOCK = True
        print("💡 Bạn chưa nhập API key -> Đang bật MOCK MODE để bạn học tiếp không bị gián đoạn.")

# Sử dụng gemini-2.5-flash tương thích hoàn toàn với google-genai SDK mới
MODEL_NAME = "gemini-3.6-flash"
print(f"🤖 Model được chỉ định: {MODEL_NAME}")

🔑 Tuyệt vời! Đã nạp thành công GEMINI_API_KEY từ Colab Secrets.
🤖 Model được chỉ định: gemini-3.6-flash


In [3]:
from google import genai
from google.genai import types

client = None

if USE_MOCK:
    print("⚠️ Đang ở chế độ Mock: Bỏ qua khởi tạo kết nối Google Client.")
else:
    try:
        # Tạo client kết nối với API Key
        client = genai.Client(api_key=api_key)
        print("🎉 Khởi tạo Google GenAI Client thành công! Đã sẵn sàng gửi yêu cầu.")
    except Exception as e:
        print("❌ Lỗi khi khởi tạo Gemini Client:", e)
        print("💡 Tự động chuyển sang Mock Mode để bảo vệ chương trình không bị crash.")
        USE_MOCK = True

🎉 Khởi tạo Google GenAI Client thành công! Đã sẵn sàng gửi yêu cầu.


In [4]:
def mock_generate_text(prompt: str, system_instruction: str = None) -> str:
    """Phản hồi giả lập để học viên vẫn học tiếp được nếu chưa có API key."""
    return (
        "[Mock Response]: Chào bạn! Đây là câu trả lời mẫu khi ở chế độ Mock. "
        "Khi có API key hợp lệ, Gemini 2.5 Flash sẽ phân tích và trả lời cực kỳ thông minh!"
    )

def generate_text(prompt: str, system_instruction: str = None) -> str:
    """
    Gửi prompt đến Gemini và trả về câu trả lời dạng văn bản.
    - prompt: Nội dung câu hỏi.
    - system_instruction: Lời chỉ dẫn vai trò/luật lệ cho AI (nếu có).
    """
    # 1. Nếu đang bật Mock mode hoặc client chưa tạo được, dùng hàm mock
    if USE_MOCK or client is None:
        return mock_generate_text(prompt, system_instruction)

    # 2. Chuẩn bị cấu hình nếu có system_instruction
    config = None
    if system_instruction:
        config = types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.3  # Độ sáng tạo vừa phải (từ 0.0 đến 2.0)
        )

    # 3. Gửi yêu cầu tới Gemini và nhận kết quả
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=config,
        )
        return response.text
    except Exception as error:
        print(f"⚠️ Không gọi được Gemini API ({error}). Chuyển sang phản hồi mock.")
        return mock_generate_text(prompt, system_instruction)


# --- THỬ NGHIỆM GỌI API LẦN ĐẦU TIÊN ---
cau_hoi_test = "Hãy giới thiệu ngắn gọn món Phở Việt Nam trong 2 câu thật cuốn hút."
print("❓ Câu hỏi gửi AI:", cau_hoi_test)

cau_tra_loi = generate_text(cau_hoi_test)
print("\n🤖 Phản hồi từ Gemini:")
print(cau_tra_loi)

❓ Câu hỏi gửi AI: Hãy giới thiệu ngắn gọn món Phở Việt Nam trong 2 câu thật cuốn hút.

🤖 Phản hồi từ Gemini:
Phở là "quốc hồn quốc túy" của ẩm thực Việt Nam, gây thương nhớ khắp thế giới bởi làn nước dùng ninh xương ngọt thanh, thơm nức hương quế, hồi và gừng nướng. 

Sự hòa quyện hoàn hảo giữa sợi bánh mềm mại, miếng thịt mỏng mọng nước và chút ớt chanh đậm đà sẽ đánh thức trọn vẹn mọi giác quan của bạn ngay từ ngụm nước dùng đầu tiên.


In [5]:
import json
from pathlib import Path
import pandas as pd

# 1. Đảm bảo có file menu.csv trên máy ảo Colab
menu_path = Path("menu.csv")
if not menu_path.exists():
    print("📁 Chưa tìm thấy 'menu.csv' trên Colab -> Đang tự động tạo dữ liệu thực đơn mẫu...")
    sample_menu_data = pd.DataFrame([
        {
            "name": "Gỏi Cuốn",
            "description": "Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá bánh tráng mềm mại, kết hợp với các loại rau sống, thịt hoặc tôm tươi, thưởng thức cùng nước mắm chua ngọt.",
            "ingredients": "bún, bánh tráng, tôm, thịt bò phi lê, rau sống",
            "notes": "Món gỏi cuốn thường được phục vụ tươi và phải ăn liền sau khi cuốn để trải nghiệm hương vị tốt nhất."
        },
        {
            "name": "Phở Việt Nam",
            "description": "Nổi tiếng với hương vị đậm đà và hương thơm của gia vị tự nhiên. Mỗi tô phở được chế biến từ nước dùng thơm ngon, kết hợp sợi bánh mềm mại và thịt bò thơm ngọt.",
            "ingredients": "bún phở, thịt bò, thịt gà, hành tây, hành phi, rau sống, giá đỗ, ớt",
            "notes": "Thịt bò có thể chọn giữa tái và chín theo ý thích của khách."
        },
        {
            "name": "Cơm Tấm",
            "description": "Món ăn đường phố trứ danh gồm cơm nấu từ gạo tấm thơm dẻo, kết hợp thịt heo nướng than hoa, trứng hấp, chả, dưa leo và nước mắm chua ngọt.",
            "ingredients": "gạo tấm, thịt heo nướng, trứng, chả, dưa leo, nước mắm chua ngọt",
            "notes": "Thích hợp cho cả bữa trưa và bữa tối no bụng."
        },
        {
            "name": "Bún Bò",
            "description": "Đặc sản ẩm thực miền Trung Việt Nam, nổi tiếng với nước dùng cay thơm nồng nàn hương sả và mắm ruốc, cùng sợi bún to dai mềm.",
            "ingredients": "bún, thịt bò, hành tây, hành tím, rau sống",
            "notes": "Thịt bò có thể chọn giữa tái, nạm, bắp bò, giò heo."
        },
        {
            "name": "Khoai Tây Chiên",
            "description": "Món ăn phổ biến với lớp vỏ giòn rụm bên ngoài và khoai mềm bùi bên trong, chấm cùng tương cà hoặc sốt cay.",
            "ingredients": "khoai tây, dầu thực vật, muối",
            "notes": "Thích hợp dùng làm món khai vị hoặc ăn kèm cho trẻ em."
        },
        {
            "name": "Phở Bò",
            "description": "Phở bò là món ăn truyền thống nổi tiếng của Việt Nam với nước dùng thơm ngon, bánh phở mềm và những lát thịt bò mỏng. Món ăn thường được dùng nóng cùng rau thơm và các loại gia vị.",
             "ingredients": "bánh phở, thịt bò, xương bò, hành lá, rau thơm, gừng, quế, hồi",
            "notes": "Phở bò ngon nhất khi được dùng nóng, có thể ăn kèm chanh, ớt và rau thơm tùy khẩu vị."
        },
        {
            "name": "Bánh Xèo",
            "description": "Bánh xèo có lớp vỏ vàng giòn được làm từ bột gạo, bên trong là nhân tôm, thịt và giá đỗ. Khi ăn thường cuốn cùng rau sống và chấm với nước mắm chua ngọt.",
            "ingredients": "bột gạo, tôm, thịt heo, giá đỗ, hành lá, rau sống, nước mắm",
            "notes": "Bánh xèo nên được thưởng thức ngay sau khi chiên để giữ được độ nóng và giòn của lớp vỏ."
        }
            ])
    sample_menu_data.to_csv(menu_path, index=False, encoding="utf-8")

# Đọc menu vào biến DataFrame
menu_df = pd.read_csv(menu_path)

# 2. Đảm bảo có file cấu hình config.json
config_path = Path("config.json")
if not config_path.exists():
    print("📁 Chưa tìm thấy 'config.json' -> Đang tự động tạo cấu hình thông tin nhà hàng...")
    sample_config = {
        "restaurant_name": "Viet Cuisine",
        "restaurant_address": "329 Scottmouth, Georgia, USA",
        "hotline": "318-237-3870",
        "functions": ["giới thiệu nhà hàng", "giới thiệu menu", "giải đáp câu hỏi về món ăn"],
        "out_of_scope_message": "Tôi đang không hỗ trợ chức năng này. Xin vui lòng liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp."
    }
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(sample_config, f, ensure_ascii=False, indent=2)

# Đọc config từ file
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print("✅ Bước 5 hoàn tất: Đã nạp dữ liệu menu và cấu hình quán ăn thành công!")
print(f"🏪 Tên quán: {config['restaurant_name']} | ☎️ Hotline: {config['hotline']}")
print(f"📊 Số lượng món ăn: {len(menu_df)} món.")

# Hiển thị bảng dữ liệu trực quan trên Google Colab
menu_df[["name", "ingredients", "notes"]]

📁 Chưa tìm thấy 'menu.csv' trên Colab -> Đang tự động tạo dữ liệu thực đơn mẫu...
📁 Chưa tìm thấy 'config.json' -> Đang tự động tạo cấu hình thông tin nhà hàng...
✅ Bước 5 hoàn tất: Đã nạp dữ liệu menu và cấu hình quán ăn thành công!
🏪 Tên quán: Viet Cuisine | ☎️ Hotline: 318-237-3870
📊 Số lượng món ăn: 7 món.


,name,ingredients,notes
0,Gỏi Cuốn,"bún, bánh tráng, tôm, thịt bò phi lê, rau sống",Món gỏi cuốn thường được phục vụ tươi và phải ...
1,Phở Việt Nam,"bún phở, thịt bò, thịt gà, hành tây, hành phi,...",Thịt bò có thể chọn giữa tái và chín theo ý th...
2,Cơm Tấm,"gạo tấm, thịt heo nướng, trứng, chả, dưa leo, ...",Thích hợp cho cả bữa trưa và bữa tối no bụng.
3,Bún Bò,"bún, thịt bò, hành tây, hành tím, rau sống","Thịt bò có thể chọn giữa tái, nạm, bắp bò, giò..."
4,Khoai Tây Chiên,"khoai tây, dầu thực vật, muối",Thích hợp dùng làm món khai vị hoặc ăn kèm cho...
5,Phở Bò,"bánh phở, thịt bò, xương bò, hành lá, rau thơm...","Phở bò ngon nhất khi được dùng nóng, có thể ăn..."
6,Bánh Xèo,"bột gạo, tôm, thịt heo, giá đỗ, hành lá, rau s...",Bánh xèo nên được thưởng thức ngay sau khi chi...


In [6]:
def build_menu_context(df: pd.DataFrame) -> str:
    """
    Chuyển đổi từng hàng dữ liệu của DataFrame thành một đoạn văn bản tóm tắt danh sách món ăn.
    """
    def clean_text(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def add_detail(current_text, label, value):
        val_clean = clean_text(value)
        if not val_clean:
            return current_text
        suffix = "" if val_clean.endswith((".", "!", "?")) else "."
        return f"{current_text} {label}: {val_clean}{suffix}"

    lines = []
    for _, row in df.iterrows():
        name = clean_text(row.get("name", ""))
        desc = clean_text(row.get("description", ""))
        ingredients = row.get("ingredients", "")
        notes = row.get("notes", "")

        # Ghép thành dòng giới thiệu món
        item_line = f"- {name}: {desc}"
        item_line = add_detail(item_line, "Thành phần", ingredients)
        item_line = add_detail(item_line, "Ghi chú", notes)
        lines.append(item_line)

    return "\n".join(lines)


# Tạo chuỗi ngữ cảnh menu và in thử
menu_context = build_menu_context(menu_df)
print("📄 Ngữ cảnh thực đơn được chuẩn bị (dạng văn bản cho AI đọc):")
print("="*70)
print(menu_context)
print("="*70)


📄 Ngữ cảnh thực đơn được chuẩn bị (dạng văn bản cho AI đọc):
- Gỏi Cuốn: Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá bánh tráng mềm mại, kết hợp với các loại rau sống, thịt hoặc tôm tươi, thưởng thức cùng nước mắm chua ngọt. Thành phần: bún, bánh tráng, tôm, thịt bò phi lê, rau sống. Ghi chú: Món gỏi cuốn thường được phục vụ tươi và phải ăn liền sau khi cuốn để trải nghiệm hương vị tốt nhất.
- Phở Việt Nam: Nổi tiếng với hương vị đậm đà và hương thơm của gia vị tự nhiên. Mỗi tô phở được chế biến từ nước dùng thơm ngon, kết hợp sợi bánh mềm mại và thịt bò thơm ngọt. Thành phần: bún phở, thịt bò, thịt gà, hành tây, hành phi, rau sống, giá đỗ, ớt. Ghi chú: Thịt bò có thể chọn giữa tái và chín theo ý thích của khách.
- Cơm Tấm: Món ăn đường phố trứ danh gồm cơm nấu từ gạo tấm thơm dẻo, kết hợp thịt heo nướng than hoa, trứng hấp, chả, dưa leo và nước mắm chua ngọt. Thành phần: gạo tấm, thịt heo nướng, trứng, chả, dưa leo, nước mắm chua ngọt. Ghi chú: Thích hợp cho cả bữa trưa và bữa tối 

In [7]:
def build_system_instruction(cfg: dict) -> str:
    """Tạo lời chỉ dẫn hệ thống định hình nhân cách và giới hạn cho PhoBot."""
    restaurant_name = cfg.get("restaurant_name", "Viet Cuisine")
    restaurant_address = cfg.get("restaurant_address", "")
    functions = ", ".join(cfg.get("functions", ["tư vấn món ăn"]))
    out_of_scope_message = cfg.get(
        "out_of_scope_message",
        "Tôi đang không hỗ trợ chức năng này. Xin vui lòng liên hệ nhân viên nhà hàng để được trợ giúp."
    )

    instructions = f"""
Bạn tên là PhoBot, một trợ lý AI thông minh, nhiệt tình và lịch sự của nhà hàng {restaurant_name}.
Địa chỉ quán: {restaurant_address}.
Các nhiệm vụ được hỗ trợ: {functions}.

Quy tắc trả lời BẮT BUỘC:
1. Luôn trả lời bằng tiếng Việt, giọng điệu vui vẻ, niềm nở và hiếu khách.
2. Chỉ tư vấn và trả lời dựa trên thông tin thực đơn trong dữ liệu được cung cấp. Tuyệt đối KHÔNG tự bịa ra món ăn không có trong danh sách.
3. Khi khách hỏi món nào đó, hãy giới thiệu thêm thành phần và ghi chú món để khách thấy hấp dẫn.
4. Nếu khách hàng hỏi những câu nằm NGOÀI phạm vi nhà hàng/ẩm thực (ví dụ: công nghệ, đồ điện tử, thời tiết, giải toán...), hãy trả lời ĐÚNG câu sau:
   "{out_of_scope_message}"
5. Giữ câu trả lời súc tích, mạch lạc, dễ đọc.
"""
    return instructions.strip()

system_instruction = build_system_instruction(config)
print("📜 Nội dung System Instruction:")
print(system_instruction)


📜 Nội dung System Instruction:
Bạn tên là PhoBot, một trợ lý AI thông minh, nhiệt tình và lịch sự của nhà hàng Viet Cuisine.
Địa chỉ quán: 329 Scottmouth, Georgia, USA.
Các nhiệm vụ được hỗ trợ: giới thiệu nhà hàng, giới thiệu menu, giải đáp câu hỏi về món ăn.

Quy tắc trả lời BẮT BUỘC:
1. Luôn trả lời bằng tiếng Việt, giọng điệu vui vẻ, niềm nở và hiếu khách.
2. Chỉ tư vấn và trả lời dựa trên thông tin thực đơn trong dữ liệu được cung cấp. Tuyệt đối KHÔNG tự bịa ra món ăn không có trong danh sách.
3. Khi khách hỏi món nào đó, hãy giới thiệu thêm thành phần và ghi chú món để khách thấy hấp dẫn.
4. Nếu khách hàng hỏi những câu nằm NGOÀI phạm vi nhà hàng/ẩm thực (ví dụ: công nghệ, đồ điện tử, thời tiết, giải toán...), hãy trả lời ĐÚNG câu sau:
   "Tôi đang không hỗ trợ chức năng này. Xin vui lòng liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp."
5. Giữ câu trả lời súc tích, mạch lạc, dễ đọc.


In [8]:
def ask_bot(question: str) -> str:
    """
    Nhận câu hỏi từ khách hàng, ghép ngữ cảnh thực đơn và gọi Gemini AI xử lý.
    """
    # Tạo prompt chứa cả dữ liệu menu và câu hỏi khách hàng
    user_prompt = f"""
Dưới đây là thông tin thực đơn của nhà hàng:
{menu_context}

Khách hàng hỏi:
"{question}"

Hãy trả lời khách hàng theo đúng vai trò PhoBot và các quy tắc đã được hướng dẫn.
"""

    # Gửi tới mô hình với system_instruction đã định hình
    return generate_text(
        prompt=user_prompt,
        system_instruction=system_instruction
    )

print("✅ Hàm ask_bot() đã sẵn sàng phục vụ!")

✅ Hàm ask_bot() đã sẵn sàng phục vụ!


In [9]:
test_questions = [
    "Quán có những loại phở nào?",
    "Một tô phở bò giá bao nhiêu?",
    "Phở nào ngon và được khách hàng gọi nhiều nhất?"
]

for idx, q in enumerate(test_questions, start=1):
    print(f"\n{'='*25} TÌNH HUỐNG THỬ NGHIỆM {idx} {'='*25}")
    print(f"👤 Khách hàng hỏi: {q}")
    tra_loi = ask_bot(q)
    print(f"\n🤖 PhoBot trả lời:\n{tra_loi}")


========================= TÌNH HUỐNG THỬ NGHIỆM 1 =========================
👤 Khách hàng hỏi: Quán có những loại phở nào?

🤖 PhoBot trả lời:
Dạ, PhoBot xin chào quý khách! Nhà hàng Viet Cuisine của chúng em hiện có 2 món phở vô cùng thơm ngon và đậm đà hương vị truyền thống sau đây ạ:

1. **Phở Việt Nam**: 
   - **Thành phần**: Bún phở, thịt bò, thịt gà, hành tây, hành phi, rau sống, giá đỗ, ớt.
   - **Ghi chú**: Món ăn nổi tiếng với nước dùng thơm ngon chế biến từ gia vị tự nhiên. Đặc biệt, quý khách có thể lựa chọn thịt bò tái hoặc chín tùy theo ý thích ạ.

2. **Phở Bò**: 
   - **Thành phần**: Bánh phở, thịt bò, xương bò, hành lá, rau thơm, gừng, quế, hồi.
   - **Ghi chú**: Phở bò chuẩn vị truyền thống với nước dùng đậm đà từ xương bò ninh cùng quế, hồi. Món này ngon nhất khi dùng nóng, ăn kèm với chanh, ớt và rau thơm ạ.

Dạ quý khách muốn chọn thưởng thức món phở nào ạ? PhoBot luôn sẵn sàng phục vụ quý khách!

========================= TÌNH HUỐNG THỬ NGHIỆM 2 =====================

In [10]:
print("="*70)
print("🍲 CHÀO MỪNG BẠN ĐẾN VỚI PHÒNG CHAT THỬ NGHIỆM PHOBOT 🍲")
print("Nhập câu hỏi của bạn và nhấn Enter. Nhập 'exit' để dừng trò chuyện.")
print("="*70)

while True:
    cau_hoi = input("\n👤 Bạn: ")
    if not cau_hoi.strip():
        continue
    if cau_hoi.strip().lower() in ["exit", "quit", "thoat", "thoát"]:
        print("\n👋 PhoBot: Cảm ơn bạn đã trò chuyện cùng PhoBot! Chúc bạn một ngày tốt lành!")
        break

    print("⏳ PhoBot đang soạn câu trả lời...")
    phan_hoi = ask_bot(cau_hoi)
    print(f"\n🤖 PhoBot:\n{phan_hoi}")

🍲 CHÀO MỪNG BẠN ĐẾN VỚI PHÒNG CHAT THỬ NGHIỆM PHOBOT 🍲
Nhập câu hỏi của bạn và nhấn Enter. Nhập 'exit' để dừng trò chuyện.

👤 Bạn: exit

👋 PhoBot: Cảm ơn bạn đã trò chuyện cùng PhoBot! Chúc bạn một ngày tốt lành!
